[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohameddddt/Speech-Emotion-Recognition/blob/main/notebooks/embeddings_colab.ipynb)

# Speech Emotion Recognition — pretrained speech embeddings

The hand-crafted feature ensemble (`project_notebook.ipynb`) reaches **53.2% accuracy** on held-out actors, and every classifier we tried landed in the 51–53% range.
That points to the features as the bottleneck.

This notebook replaces them with embeddings from **frozen self-supervised speech encoders** (WavLM, HuBERT), which were pretrained on thousands of hours of speech:

1. Run every clip through the encoder and mean-pool the hidden states of **every layer** over time. The result is cached, so it runs only once per encoder.
2. Train a **linear probe** (standardise + logistic regression). The layer(s) and regularisation are chosen with actor-grouped CV on the training actors.
3. Score once on **the same 19 held-out actors** as the baseline, so the numbers are directly comparable.

The encoder is never fine-tuned here.

**Result from a local run:** WavLM-base-plus reaches **74.2% accuracy / 0.742 macro F1** on the held-out actors, **21 points above the baseline**. Its CV estimate on the training actors was also 74.2%.

**Expected runtime (T4 GPU, not yet measured on Colab):** a few minutes for the setup cell, a few minutes of extraction per base encoder, and several minutes per probe selection, which runs on Colab's 2 CPU cores. On an 8-core laptop CPU, the probe selection took ~4.5 min.

## 0. Setup

Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*. The notebook also runs on CPU, just much slower.

Embeddings are written to Google Drive if you mount it, so a disconnect or a new session doesn't lose them.
Extraction is also checkpointed every 500 clips and resumes where it stopped.

In [ ]:
import os, sys, subprocess

REPO_DIR = "/content/Speech-Emotion-Recognition"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/mohameddddt/Speech-Emotion-Recognition.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only  # pick up code changes pushed since the clone
%cd {REPO_DIR}
!git log --oneline -1
if not os.path.exists("ser/embeddings.py"):
    raise RuntimeError("This checkout has no ser/embeddings.py; push the latest code to GitHub, "
                       "then Runtime -> Restart session and run again.")
!pip install -q "librosa>=0.10" "soundfile>=0.12"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime — extraction will be slow)")

In [ ]:
USE_DRIVE = True  # keep embeddings in Google Drive across sessions

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = "/content/drive/MyDrive/SER/embeddings"
    os.makedirs(drive_dir, exist_ok=True)
    os.makedirs("artifacts", exist_ok=True)
    if not os.path.islink("artifacts/embeddings"):
        !rm -rf artifacts/embeddings
        os.symlink(drive_dir, "artifacts/embeddings")

## 1. Configuration

| encoder | params | embedding size |
|---|---|---|
| `wavlm-base-plus` | 95M | 13 layers × 768 |
| `hubert-base` | 95M | 13 layers × 768 |
| `wav2vec2-base` | 95M | 13 layers × 768 |
| `wavlm-large` | 317M | 25 layers × 1024 |

The base encoders take minutes on a T4; `wavlm-large` takes roughly 3× longer. Add `"wavlm-large"` for the strongest encoder.
The elapsed time for each encoder is printed below.

In [ ]:
ENCODERS_TO_RUN = ["wavlm-base-plus", "hubert-base"]
BASELINE_ACCURACY = 0.532  # hand-crafted feature ensemble, same held-out actors

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.metrics import ConfusionMatrixDisplay

from ser.data import DATA_DIR, speaker_split
from ser.features import build_metadata
from ser.embeddings import extract_embeddings, default_device
from train_embeddings import run

meta = build_metadata(DATA_DIR)
print(len(meta), "clips  | device:", default_device())

## 2. Extract embeddings

In [ ]:
for name in ENCODERS_TO_RUN:
    t = time.time()
    E = extract_embeddings(list(meta["path"]), name)
    print(f"{name}: {E.shape} (clips, layers + 1, dim) in {time.time() - t:.0f}s")

## 3. Linear probe per encoder

For each encoder this scores every layer with 4-fold actor-grouped CV, picks the best layer window and `C`, then evaluates once on the held-out actors.

In [ ]:
results, models = {}, {}
for name in ENCODERS_TO_RUN:
    print(f"==================== {name} ====================")
    models[name], results[name] = run(name)

## 4. Results

In [ ]:
table = pd.DataFrame([
    {"model": "hand-crafted ensemble (baseline)", "accuracy": BASELINE_ACCURACY},
    *[{"model": f"{n} probe (layers {r['layers']})", "accuracy": r["accuracy"], "macro F1": r["macro_f1"],
       "CV accuracy": round(r["cv_accuracy"], 3)} for n, r in results.items()],
]).set_index("model")
table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for name, r in results.items():
    ax.plot(r["layer_cv_scores"], marker="o", label=name)
ax.axhline(BASELINE_ACCURACY, ls="--", c="gray", label="hand-crafted baseline (test)")
ax.set_xlabel("layer (0 = CNN features)"); ax.set_ylabel("CV accuracy"); ax.legend()
ax.set_title("Which layer carries emotion? (linear probe, actor-grouped CV)")
plt.tight_layout()

Emotion information peaks in the **middle layers**: for WavLM-base-plus, single layers 5–9 all score ~70% CV accuracy, and averaging them gives ~74%. The lowest layers are close to raw acoustics, while the top layers specialise towards the pretraining objective (predicting masked speech units).

In [ ]:
best = max(results, key=lambda n: results[n]["accuracy"])
r = results[best]
cm = np.array(r["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm / cm.sum(axis=1, keepdims=True), display_labels=r["labels"]).plot(
    ax=ax, values_format=".2f", colorbar=False)
ax.set_title(f"{best}: recall per class (held-out actors)"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
print(f"best encoder: {best}  accuracy {r['accuracy']:.3f} vs baseline {BASELINE_ACCURACY:.3f} "
      f"({(r['accuracy'] - BASELINE_ACCURACY) * 100:+.1f} points)")

## 5. Try your own recording

The encoder runs on each uploaded file, so the first prediction also downloads and loads the model.

In [ ]:
from google.colab import files

model = models[best]
for name in files.upload():
    label, scores = model.predict_file(name)
    ranked = ", ".join(f"{k} {v:.2f}" for k, v in sorted(scores.items(), key=lambda kv: -kv[1]))
    print(f"{name}: {label}  ({ranked})")
    display(Audio(name))

## 6. Download the trained probe (optional)

The file holds only the encoder name, the chosen layers and the logistic regression, so it's small. The encoder weights are re-downloaded from Hugging Face when needed.
Use it locally with `python predict.py --model embed_<encoder>.joblib file.wav` (requires `torch` and `transformers`).

In [ ]:
files.download(f"artifacts/embed_{best}.joblib")
files.download(f"artifacts/embed_{best}.json")